In [1]:
import pandas as pd
import sys

sys.path.append('..')

eval_tox21 = pd.read_csv('../data/toxicity.txt',sep='\t')

In [2]:
# 检查'nontoxic'列是否有空值
has_null = eval_tox21['nontoxic'].isnull().sum()
print(f"'nontoxic'列空值: {has_null}")
eval_tox21.head()

'nontoxic'列空值: 13


,CASRN,DTXSID,Chemical_Name,Structure_Source,Canonical_QSARr,Salt_Solvent,InChI_Code_QSARr,InChI_Key_QSARr,very_toxic,nontoxic,LD50_mgkg,EPA_category,GHS_category
0,100-00-5,DTXSID5020281,1-Chloro-4-nitrobenzene,EPA_DSSTox,[O-][N+](=O)C1C=CC(Cl)=CC=1,?,InChI=1S/C6H4ClNO2/c7-5-1-3-6(4-2-5)8(9)10/h1-4H,CZGCEKJOLUNIFY-UHFFFAOYSA-N,False,False,460.0,2.0,4.0
1,100-01-6,DTXSID8020961,4-Nitroaniline,EPA_DSSTox,NC1=CC=C(C=C1)[N+]([O-])=O,?,InChI=1S/C6H6N2O2/c7-5-1-3-6(4-2-5)8(9)10/h1-4...,TYMLOMAKGOJONV-UHFFFAOYSA-N,False,False,750.0,3.0,4.0
2,100-02-7,DTXSID0021834,4-Nitrophenol,EPA_DSSTox,[O-][N+](=O)C1C=CC(O)=CC=1,?,"InChI=1S/C6H5NO3/c8-6-3-1-5(2-4-6)7(9)10/h1-4,8H",BTJIUGUIPKRLHP-UHFFFAOYSA-N,False,False,170.0,2.0,3.0
3,100-14-1,DTXSID4025745,4-Nitrobenzyl chloride,EPA_DSSTox,[O-][N+](=O)C1C=CC(CCl)=CC=1,?,InChI=1S/C7H6ClNO2/c8-5-6-1-3-7(4-2-6)9(10)11/...,KGCNHWXDPDPSBV-UHFFFAOYSA-N,False,False,1809.0,3.0,4.0
4,100-15-2,DTXSID7025635,N-Methyl-4-nitroaniline,EPA_DSSTox,CNC1C=CC(=CC=1)[N+]([O-])=O,?,InChI=1S/C7H8N2O2/c1-8-6-2-4-7(5-3-6)9(10)11/h...,XIFJZJPMHNUGRA-UHFFFAOYSA-N,False,True,NaN,3.0,5.0


In [3]:
from util.rdkit_util import *

# 只保留 nontoxic 标签不为空的行
eval_tox21_valid = eval_tox21[eval_tox21['nontoxic'].notnull()].copy()

# 从筛选后的子集里取出smiles和label，保证一一对应
eval_tox21_smiles = eval_tox21_valid['Canonical_QSARr']
eval_tox21_label = eval_tox21_valid['nontoxic'].astype(int)

print(f"有效样本数量：{len(eval_tox21_smiles)}, label数量：{len(eval_tox21_label)}")


有效样本数量：8981, label数量：8981


In [4]:
from util.chemixnet_model_util import *
from util.eval_util import *

valid_tox21_smiles, valid_tox21_maccs, valid_tox21_label = get_maccs_key(eval_tox21_smiles, eval_tox21_label)

[10:42:08] Explicit valence for atom # 1 Si, 5, is greater than permitted


In [5]:
print(valid_tox21_smiles[0])
print(valid_tox21_maccs[0])
print(valid_tox21_label[0])

[O-][N+](=O)C1C=CC(Cl)=CC=1
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 1 0 1 1 0 1 1 1 1 1 0]
0


In [6]:
from util.dl_util import *

tox21_smiles = []

for smile in valid_tox21_smiles:
    tox21_smiles += [len(smile)]

max_len_tox21 = max(tox21_smiles)
vocab_size = generate_math_code_tokenizer(valid_tox21_smiles, encode_type = 'vocab_size' )

In [7]:
from keras.preprocessing import sequence

embedding_tox21 = generate_math_code_tokenizer(valid_tox21_smiles)
tox_smiles_sequence = sequence.pad_sequences(embedding_tox21)

In [8]:
x1 = tox_smiles_sequence
x2 = valid_tox21_maccs
y = valid_tox21_label

In [9]:
y_test, y_pre = split_data_and_train_model(CNN_MODEL,x1,x2,y, vocab_size, max_len_tox21, 'evol_tox21',batch_size=32, epochs=100)

CNN_MODEL


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 940, 128)       │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 938, 512)       │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 469, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 467, 256)       │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 233, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 231, 128)       │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 115, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 113, 64)        │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 56, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 719,105 (2.74 MB)

 Trainable params: 719,105 (2.74 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
 46/203 ━━━━━━━━━━━━━━━━━━━━ 27s 173ms/step - loss: 0.2466 - mae: 0.4924

KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import GridSearchCV
from keras.wrappers.scikit_learn import KerasClassifier

# 定义一个构建模型的函数，供KerasClassifier调用
def create_model(filters=32, kernel_size=3, dropout_rate=0.5):
    # 这里假设CNN_MODEL是一个函数/类，接受这些参数
    # 你可能需要按照自己的模型结构进行调整
    return CNN_MODEL(
        vocab_size=vocab_size,
        max_len=max_len_tox21,
        filters=filters,
        kernel_size=kernel_size,
        dropout_rate=dropout_rate
    )

# 构造KerasClassifier包装器
model = KerasClassifier(build_fn=create_model, 
                       x2=x2,  # 如果你的模型需要第二输入，在create_model中处理
                       epochs=100, 
                       batch_size=32, 
                       verbose=0)

# 网格搜索参数
param_grid = {
    'filters': [32, 64],
    'kernel_size': [3, 5],
    'dropout_rate': [0.3, 0.5, 0.7],
    'batch_size': [16, 32]
}

# 执行网格搜索
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
grid_result = grid.fit(x1, y)

# 打印最佳参数和分数
print("Best parameters found: ", grid_result.best_params_)
print("Best cross-validation score: {:.4f}".format(grid_result.best_score_))